# Clase 098 — Gaussian Mixture Models

GMM como *soft clustering* probabilístico ajustado con EM: `predict_proba`, elección de componentes con BIC/AIC, `covariance_type` y detección de anomalías por densidad.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

## 1. Ajuste básico con EM

`GaussianMixture` asume mezcla de gaussianas. Comparamos `predict` contra las etiquetas reales con ARI.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score

np.random.seed(42)
X, y_true = make_blobs(n_samples=1000, centers=3, cluster_std=[1.0, 2.0, 0.5], random_state=42)

gmm = GaussianMixture(n_components=3, n_init=10, random_state=42).fit(X)
labels = gmm.predict(X)
print("ARI GMM vs verdad:", round(adjusted_rand_score(y_true, labels), 4))
print("pesos de mezcla:", gmm.weights_.round(3))
assert gmm.converged_, "EM deberia converger"

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=10, alpha=0.6)
plt.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c="black", marker="X", s=150)
plt.title("GMM (3 componentes) sobre blobs de varianza distinta")
plt.tight_layout(); plt.show()

## 2. Soft clustering: `predict_proba`

A diferencia de K-Means (asignación dura), GMM da un vector de probabilidades por punto. Cerca de la frontera las probas se reparten.

In [ ]:
proba = gmm.predict_proba(X)
incertidumbre = 1 - proba.max(axis=1)
frontera = np.argsort(incertidumbre)[-5:]  # 5 puntos mas ambiguos

print("probabilidades de los 5 puntos mas ambiguos:")
for i in frontera:
    print("  ", proba[i].round(3), "-> incertidumbre", round(incertidumbre[i], 3))
assert incertidumbre.max() > 0.1, "algun punto deberia estar cerca de la frontera"
print("GMM reparte probabilidad; K-Means asignaria duro.")

## 3. Selección de componentes con BIC y AIC

Ambos penalizan la complejidad. Elegimos el `n_components` con **BIC mínimo**.

In [ ]:
ns = range(1, 9)
bics, aics = [], []
for n in ns:
    g = GaussianMixture(n_components=n, n_init=5, random_state=42).fit(X)
    bics.append(g.bic(X)); aics.append(g.aic(X))

n_bic = list(ns)[int(np.argmin(bics))]
print("n_components optimo por BIC:", n_bic)
for n, b, a in zip(ns, bics, aics):
    print(f"n={n}: BIC={b:.0f}  AIC={a:.0f}")
assert n_bic in (2, 3, 4), "el BIC deberia elegir pocos componentes"

plt.figure(figsize=(7, 4))
plt.plot(list(ns), bics, "o-", label="BIC", color="#37a")
plt.plot(list(ns), aics, "s-", label="AIC", color="#3a7")
plt.axvline(n_bic, ls="--", color="#c33", lw=0.8, label=f"BIC min (n={n_bic})")
plt.xlabel("n_components"); plt.ylabel("criterio")
plt.title("Seleccion de K con BIC / AIC")
plt.legend(); plt.tight_layout(); plt.show()

## 4. `covariance_type`

`full`, `tied`, `diag`, `spherical` cambian la flexibilidad y el número de parámetros. Comparamos BIC.

In [ ]:
print(f"{'cov_type':>10} {'BIC':>10}")
for cov in ("full", "tied", "diag", "spherical"):
    g = GaussianMixture(n_components=3, covariance_type=cov, n_init=5, random_state=42).fit(X)
    print(f"{cov:>10} {g.bic(X):>10.0f}")
print("\n'full' es el mas flexible; 'spherical' el mas parsimonioso.")

## 5. Bayesian GMM: apaga componentes sobrantes

Con `n_components=10` y prior de Dirichlet chico, los pesos efectivos colapsan a ~3 (los clusters reales).

In [ ]:
from sklearn.mixture import BayesianGaussianMixture

bgm = BayesianGaussianMixture(n_components=10, weight_concentration_prior=0.01,
                              n_init=5, max_iter=300, random_state=42).fit(X)
pesos_efectivos = int((bgm.weights_ > 0.05).sum())
print("pesos ordenados:", np.sort(bgm.weights_)[::-1].round(3))
print("componentes con peso > 0.05:", pesos_efectivos)
assert pesos_efectivos <= 4, "el Bayesian GMM deberia apagar los sobrantes"

plt.figure(figsize=(7, 4))
plt.bar(range(1, 11), np.sort(bgm.weights_)[::-1], color="#37a")
plt.axhline(0.05, ls="--", color="#c33", lw=0.8, label="umbral 0.05")
plt.xlabel("componente"); plt.ylabel("peso efectivo")
plt.title("Bayesian GMM apaga los componentes innecesarios")
plt.legend(); plt.tight_layout(); plt.show()

## 6. Detección de anomalías por densidad

`score_samples` da la log-densidad; los puntos bajo un percentil bajo son anomalías.

In [ ]:
scores = gmm.score_samples(X)
umbral = np.percentile(scores, 4)
anomalias = scores < umbral
print(f"umbral (percentil 4): {umbral:.2f} | anomalias: {int(anomalias.sum())}")

plt.figure(figsize=(7, 5))
plt.scatter(X[~anomalias, 0], X[~anomalias, 1], c="#37a", s=8, label="normal")
plt.scatter(X[anomalias, 0], X[anomalias, 1], c="#c33", s=30, marker="x", label="anomalia")
plt.legend(); plt.title("Anomalias por baja log-densidad (GMM)")
plt.tight_layout(); plt.show()

## Ejercicios

1. Ajustá `GaussianMixture(n_components=3)` sobre blobs de `cluster_std` variable y compará `predict` con ARI.
2. Mostrá `predict_proba` de 5 puntos de frontera y contrastá con la asignación dura de K-Means.
3. Ajustá GMMs con `n_components` de 1 a 10, graficá BIC/AIC y encontrá el mínimo.
4. Repetí con los 4 `covariance_type` y compará BIC.
5. Ajustá `BayesianGaussianMixture(n_components=10, weight_concentration_prior=0.01)` y verificá que los pesos efectivos son ≈ 3.

## Conclusiones

- GMM hace *soft clustering*: da probabilidades de pertenencia, no una etiqueta dura como K-Means.
- Elegí `n_components` con BIC/AIC (nunca con la log-verosimilitud, que siempre sube con K).
- `covariance_type` controla el trade-off flexibilidad/parámetros: `full` sobreajusta con pocos datos.
- `score_samples` habilita detección de anomalías por densidad; Bayesian GMM elige K solo apagando componentes.

## ✅ Soluciones de los ejercicios

Cinco ejercicios de GMM: ajuste y ARI, asignación blanda vs dura, selección de K por BIC/AIC, `covariance_type` y la GMM bayesiana que apaga componentes sobrantes. `n_jobs=1`.

**Ejercicio 1 — Ajuste básico.** `GaussianMixture(n_components=3)` sobre 3 blobs; comparamos con las etiquetas reales vía ARI.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

X, ytrue = make_blobs(n_samples=600, centers=3, cluster_std=[1.0, 2.5, 0.6],
                      random_state=42)
gmm = GaussianMixture(n_components=3, random_state=42).fit(X)
lab = gmm.predict(X)
print('ARI GMM vs etiquetas reales:', round(adjusted_rand_score(ytrue, lab), 3))
assert adjusted_rand_score(ytrue, lab) > 0.6

**Ejercicio 2 — Soft vs hard.** `predict_proba` de puntos en la frontera: GMM reparte responsabilidad; K-Means asigna duro.

In [ ]:
proba = gmm.predict_proba(X)
incert = 1 - proba.max(1)
border = np.argsort(incert)[-5:]  # los 5 mas ambiguos
print('probabilidades de los 5 puntos mas ambiguos:')
print(np.round(proba[border], 3))
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
print('\nK-Means (asignacion dura) para esos puntos:', km.labels_[border])
print('GMM expresa la duda con probabilidades; K-Means la esconde.')

**Ejercicio 3 — Selección de K con BIC/AIC.** Ambos penalizan complejidad; el mínimo indica el K razonable.

In [ ]:
Ks = range(1, 11)
bic = [GaussianMixture(n_components=k, random_state=42).fit(X).bic(X) for k in Ks]
aic = [GaussianMixture(n_components=k, random_state=42).fit(X).aic(X) for k in Ks]
plt.figure(figsize=(6, 4))
plt.plot(list(Ks), bic, 'o-', label='BIC'); plt.plot(list(Ks), aic, 's-', label='AIC')
plt.xlabel('n_components'); plt.legend(); plt.title('BIC / AIC vs K')
plt.tight_layout(); plt.show()
print('K minimo por BIC:', list(Ks)[int(np.argmin(bic))])

**Ejercicio 4 — `covariance_type`.** Sobre clusters elípticos rotados, `full` captura la orientación; comparamos BIC de los 4 tipos.

In [ ]:
rng = np.random.default_rng(42)
A = rng.normal(size=(2, 2))
Xe = np.vstack([rng.normal([0, 0], 1, (200, 2)) @ A,
                rng.normal([6, 6], 1, (200, 2)) @ A])
for ct in ['spherical', 'diag', 'tied', 'full']:
    g = GaussianMixture(n_components=2, covariance_type=ct, random_state=42).fit(Xe)
    print(f'  covariance_type={ct:<9} -> BIC {g.bic(Xe):.1f}')
print('full permite elipses rotadas: mejor BIC en clusters con covarianza inclinada.')

**Ejercicio 5 — GMM bayesiana.** Con `weight_concentration_prior` chico, apaga los componentes de más: 3 clusters reales → ~3 pesos efectivos.

In [ ]:
bgm = BayesianGaussianMixture(n_components=10, weight_concentration_prior=0.01,
                              random_state=42, max_iter=300).fit(X)
efectivos = int((bgm.weights_ > 0.02).sum())
print('pesos (redondeados):', np.round(bgm.weights_, 3))
print('componentes efectivos (peso > 0.02):', efectivos)
assert 2 <= efectivos <= 4, 'deberia converger a ~3 componentes'
print('OK: el prior bayesiano poda los componentes sobrantes automaticamente.')